<a href="https://colab.research.google.com/github/VinayaSharada/FinancialAnalyticsCourse/blob/main/CCCustomerLTV/clv_analysis_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Credit Card Customer Lifetime Value Analysis

This notebook demonstrates comprehensive Customer Lifetime Value (CLV) analysis for credit card customers, including predictive modeling, customer segmentation, and actionable business insights.

## 📋 Learning Objectives
- Understand different CLV calculation methodologies
- Implement customer segmentation strategies
- Build predictive models for customer value
- Generate actionable insights for bank management

## 🏦 Business Impact
CLV analysis enables banks to:
- Optimize customer acquisition costs
- Identify high-value customers for retention
- Develop targeted marketing strategies
- Make data-driven pricing decisions
- Improve portfolio profitability

## 1. Setup and Data Loading

In [ ]:
# Install required packages (run this cell in Google Colab)
!pip install pandas numpy matplotlib seaborn scikit-learn faker

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import warnings
import os

# Configure plotting and warnings
plt.style.use('seaborn-v0_8')
warnings.filterwarnings('ignore')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ All packages imported successfully!")

## 2. Load Data from GitHub

In [ ]:
# Load synthetic data from GitHub repository
import urllib.request

data_url = "https://raw.githubusercontent.com/VinayaSharada/FinancialAnalyticsCourse/main/CCCustomerLTV/syntheticdata.csv"

try:
    # Try to download from GitHub
    urllib.request.urlretrieve(data_url, "syntheticdata.csv")
    print("✅ Downloaded syntheticdata.csv from GitHub")
    df = pd.read_csv("syntheticdata.csv")
except Exception as e:
    print(f"⚠️ Could not download from GitHub: {e}")
    print("Please ensure you have generated the data using the data generation notebook first.")
    # For local development, try to load from current directory
    try:
        df = pd.read_csv("syntheticdata.csv")
        print("✅ Loaded syntheticdata.csv from local directory")
    except FileNotFoundError:
        print("❌ syntheticdata.csv not found. Please run the data generation notebook first.")
        raise

print(f"📊 Dataset loaded: {len(df):,} records with {len(df.columns)} features")

## 3. Data Overview and Validation

In [ ]:
# Display basic information about the dataset
print("📋 DATASET OVERVIEW")
print("=" * 50)
print(f"Shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\n🔍 SAMPLE DATA")
print("=" * 50)
display(df.head())

print("\n📊 DATA TYPES")
print("=" * 50)
print(df.dtypes.value_counts())

In [ ]:
# Key statistics
print("📈 KEY STATISTICS")
print("=" * 50)

key_metrics = ['monthly_profit', 'monthly_revenue', 'monthly_cost', 'avg_monthly_spend', 'tenure_months']
display(df[key_metrics].describe())

print("\n💰 PROFITABILITY BREAKDOWN")
print("-" * 30)
profitable = df[df['monthly_profit'] > 0]
unprofitable = df[df['monthly_profit'] <= 0]

print(f"Profitable customers: {len(profitable):,} ({len(profitable)/len(df)*100:.1f}%)")
print(f"Unprofitable customers: {len(unprofitable):,} ({len(unprofitable)/len(df)*100:.1f}%)")
print(f"Total monthly profit: ₹{df['monthly_profit'].sum():,.2f}")

## 4. Customer Lifetime Value Calculations

In [ ]:
# Configuration for CLV calculations
CLV_CONFIG = {
    'discount_rate': 0.10,  # Annual discount rate
    'prediction_months': 36,  # Forecast period
    'base_churn_rate': 0.15  # Base churn probability
}

print("⚙️ CLV CALCULATION CONFIGURATION")
print("=" * 50)
for key, value in CLV_CONFIG.items():
    print(f"{key}: {value}")

In [ ]:
def estimate_churn_probability(df):
    """Estimate churn probability based on customer characteristics"""
    churn_prob = CLV_CONFIG['base_churn_rate']  # Base churn rate
    
    # Increase churn probability based on risk factors
    churn_factors = (
        (df['utilization_rate'] > 0.9) * 0.3 +  # High utilization
        (df['late_payments_12m'] > 6) * 0.4 +    # Frequent late payments
        (df['payment_ratio'] < 0.2) * 0.2 +      # Low payment ratio
        (df['monthly_profit'] <= 0) * 0.5 +      # Unprofitable customers
        (~df['is_active']) * 0.8                 # Inactive customers
    )
    
    churn_probability = np.clip(churn_prob + churn_factors, 0.01, 0.95)
    return churn_probability

def calculate_npv_clv(df, discount_rate, prediction_months):
    """Calculate NPV-based CLV"""
    monthly_discount_rate = discount_rate / 12
    monthly_cashflows = df['monthly_profit']
    
    # Calculate survival rates and discount factors
    survival_rates = (1 - df['churn_probability']) ** np.arange(1, prediction_months + 1).reshape(-1, 1)
    discount_factors = (1 + monthly_discount_rate) ** (-np.arange(1, prediction_months + 1).reshape(-1, 1))
    
    # Calculate NPV for each customer
    npv_clv = np.sum(monthly_cashflows.values * survival_rates * discount_factors, axis=0)
    return npv_clv

# Calculate CLV metrics
print("🧮 CALCULATING CLV METRICS...")
print("=" * 50)

# 1. Historical CLV (based on tenure)
df['historical_clv'] = df['monthly_profit'] * df['tenure_months']
print("✅ Historical CLV calculated")

# 2. Estimate churn probability
df['churn_probability'] = estimate_churn_probability(df)
print("✅ Churn probability estimated")

# 3. Simple CLV (current monthly profit * predicted lifetime)
df['estimated_lifetime_months'] = 1 / (df['churn_probability'] + 0.001)  # Avoid division by zero
df['estimated_lifetime_months'] = np.clip(df['estimated_lifetime_months'], 1, 120)
df['simple_clv'] = df['monthly_profit'] * df['estimated_lifetime_months']
print("✅ Simple CLV calculated")

# 4. NPV-based CLV
df['npv_clv'] = calculate_npv_clv(df, CLV_CONFIG['discount_rate'], CLV_CONFIG['prediction_months'])
print("✅ NPV-based CLV calculated")

print(f"\n📊 CLV Summary Statistics:")
clv_columns = ['historical_clv', 'simple_clv', 'npv_clv']
display(df[clv_columns].describe())

## 5. Predictive CLV Model

In [ ]:
# Build predictive CLV model using machine learning
print("🤖 BUILDING PREDICTIVE CLV MODEL")
print("=" * 50)

# Features for CLV prediction
feature_cols = ['age', 'annual_income', 'tenure_months', 'credit_limit', 'avg_monthly_spend',
               'utilization_rate', 'payment_ratio', 'late_payments_12m', 'monthly_revenue']

# Prepare features
X = df[feature_cols].copy()

# Add categorical features
if 'card_type' in df.columns:
    card_type_dummies = pd.get_dummies(df['card_type'], prefix='card_type')
    X = pd.concat([X, card_type_dummies], axis=1)

# Target variable (using simple CLV as target)
y = df['simple_clv']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Features: {len(X.columns)}")

# Train Random Forest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Make predictions
y_pred_train = rf_model.predict(X_train)
y_pred_test = rf_model.predict(X_test)
df['predictive_clv'] = rf_model.predict(X)

# Evaluate model
train_mse = mean_squared_error(y_train, y_pred_train)
test_mse = mean_squared_error(y_test, y_pred_test)
train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)

print(f"\n📊 Model Performance:")
print(f"Training R²: {train_r2:.3f}")
print(f"Test R²: {test_r2:.3f}")
print(f"Training RMSE: ₹{np.sqrt(train_mse):,.2f}")
print(f"Test RMSE: ₹{np.sqrt(test_mse):,.2f}")

print("✅ Predictive CLV model trained successfully!")

In [ ]:
# Feature importance analysis
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

# Plot feature importance
plt.figure(figsize=(12, 8))
top_features = feature_importance.head(10)
sns.barplot(data=top_features, y='feature', x='importance', palette='viridis')
plt.title('Top 10 Most Important Features for CLV Prediction', fontsize=14, fontweight='bold')
plt.xlabel('Feature Importance')
plt.ylabel('Features')
plt.tight_layout()
plt.show()

print("🔍 TOP 10 MOST IMPORTANT FEATURES:")
print("=" * 50)
for i, (_, row) in enumerate(top_features.iterrows(), 1):
    print(f"{i:2d}. {row['feature']:<25} {row['importance']:.4f}")

## 6. Customer Segmentation

In [ ]:
# Perform customer segmentation based on CLV and behavior
print("🎯 PERFORMING CUSTOMER SEGMENTATION")
print("=" * 50)

# Features for segmentation
segmentation_features = ['simple_clv', 'monthly_profit', 'avg_monthly_spend', 
                        'tenure_months', 'churn_probability']

# Prepare data for clustering
X_segment = df[segmentation_features].copy()

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_segment)

# Determine optimal number of clusters
inertias = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

# Plot elbow curve
plt.figure(figsize=(10, 6))
plt.plot(k_range, inertias, 'bo-')
plt.title('Elbow Method for Optimal Number of Clusters')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.grid(True, alpha=0.3)
plt.show()

# Use 5 clusters based on business logic
n_clusters = 5
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
segments = kmeans.fit_predict(X_scaled)

# Add segments to dataframe
df['segment'] = segments

print(f"✅ Customers segmented into {n_clusters} groups")

In [ ]:
# Analyze segments and assign meaningful names
segment_summary = df.groupby('segment').agg({
    'simple_clv': 'mean',
    'monthly_profit': 'mean',
    'churn_probability': 'mean',
    'avg_monthly_spend': 'mean',
    'tenure_months': 'mean',
    'customer_id': 'count'
}).round(2)

segment_summary.columns = ['avg_clv', 'avg_monthly_profit', 'avg_churn_prob', 
                          'avg_monthly_spend', 'avg_tenure', 'customer_count']

# Assign meaningful names based on characteristics
segment_names = {}
for seg in segment_summary.index:
    row = segment_summary.loc[seg]
    if row['avg_clv'] > segment_summary['avg_clv'].quantile(0.8):
        if row['avg_churn_prob'] < 0.2:
            segment_names[seg] = 'Champions'
        else:
            segment_names[seg] = 'High Value at Risk'
    elif row['avg_monthly_profit'] > segment_summary['avg_monthly_profit'].quantile(0.6):
        segment_names[seg] = 'Loyal Customers'
    elif row['avg_churn_prob'] > 0.4:
        segment_names[seg] = 'At Risk'
    else:
        segment_names[seg] = 'Potential Loyalists'

# Ensure all segments have unique names
used_names = set()
for seg in segment_summary.index:
    if segment_names[seg] in used_names:
        segment_names[seg] = f'Segment_{seg + 1}'
    used_names.add(segment_names[seg])

df['segment_name'] = df['segment'].map(segment_names)

# Display segment summary
segment_summary['segment_name'] = segment_summary.index.map(segment_names)
segment_summary = segment_summary.set_index('segment_name')

print("📊 CUSTOMER SEGMENT ANALYSIS")
print("=" * 70)
display(segment_summary.sort_values('avg_clv', ascending=False))

## 7. Comprehensive Visualizations

In [ ]:
# CLV Analysis Dashboard
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Customer Lifetime Value Analysis Dashboard', fontsize=16, fontweight='bold')

# CLV Distribution
axes[0,0].hist(df['simple_clv'], bins=50, alpha=0.7, color='skyblue', edgecolor='black')
axes[0,0].set_title('Distribution of Customer Lifetime Value')
axes[0,0].set_xlabel('CLV (₹)')
axes[0,0].set_ylabel('Number of Customers')
axes[0,0].axvline(df['simple_clv'].mean(), color='red', linestyle='--', 
                 label=f'Mean: ₹{df["simple_clv"].mean():,.0f}')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# CLV vs Monthly Profit (colored by churn risk)
scatter = axes[0,1].scatter(df['monthly_profit'], df['simple_clv'], 
                          alpha=0.6, c=df['churn_probability'], cmap='RdYlBu_r')
axes[0,1].set_title('CLV vs Monthly Profit (colored by Churn Risk)')
axes[0,1].set_xlabel('Monthly Profit (₹)')
axes[0,1].set_ylabel('CLV (₹)')
plt.colorbar(scatter, ax=axes[0,1], label='Churn Probability')
axes[0,1].grid(True, alpha=0.3)

# Customer Segments Distribution
segment_counts = df['segment_name'].value_counts()
axes[1,0].pie(segment_counts.values, labels=segment_counts.index, autopct='%1.1f%%', startangle=90)
axes[1,0].set_title('Customer Segmentation Distribution')

# CLV by Segment
sns.boxplot(data=df, x='segment_name', y='simple_clv', ax=axes[1,1])
axes[1,1].set_title('CLV Distribution by Customer Segment')
axes[1,1].set_xlabel('Customer Segment')
axes[1,1].set_ylabel('CLV (₹)')
axes[1,1].tick_params(axis='x', rotation=45)
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Segment Analysis Dashboard
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Customer Segment Analysis Dashboard', fontsize=16, fontweight='bold')

# Average CLV by Segment
segment_clv = df.groupby('segment_name')['simple_clv'].mean().sort_values(ascending=False)
axes[0,0].bar(segment_clv.index, segment_clv.values, color='lightcoral')
axes[0,0].set_title('Average CLV by Segment')
axes[0,0].set_ylabel('Average CLV (₹)')
axes[0,0].tick_params(axis='x', rotation=45)
axes[0,0].grid(True, alpha=0.3)

# Customer Count by Segment
segment_counts = df['segment_name'].value_counts()
axes[0,1].bar(segment_counts.index, segment_counts.values, color='lightgreen')
axes[0,1].set_title('Customer Count by Segment')
axes[0,1].set_ylabel('Number of Customers')
axes[0,1].tick_params(axis='x', rotation=45)
axes[0,1].grid(True, alpha=0.3)

# Churn Risk by Segment
segment_churn = df.groupby('segment_name')['churn_probability'].mean().sort_values(ascending=False)
axes[1,0].bar(segment_churn.index, segment_churn.values, color='orange')
axes[1,0].set_title('Average Churn Risk by Segment')
axes[1,0].set_ylabel('Churn Probability')
axes[1,0].tick_params(axis='x', rotation=45)
axes[1,0].grid(True, alpha=0.3)

# Monthly Profit by Segment
segment_profit = df.groupby('segment_name')['monthly_profit'].mean().sort_values(ascending=False)
axes[1,1].bar(segment_profit.index, segment_profit.values, color='skyblue')
axes[1,1].set_title('Average Monthly Profit by Segment')
axes[1,1].set_ylabel('Monthly Profit (₹)')
axes[1,1].tick_params(axis='x', rotation=45)
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Model validation plot
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('CLV Prediction Model Validation', fontsize=14, fontweight='bold')

# Actual vs Predicted (Training)
axes[0].scatter(y_train, y_pred_train, alpha=0.6, color='blue')
axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2)
axes[0].set_title(f'Training Set (R² = {train_r2:.3f})')
axes[0].set_xlabel('Actual CLV (₹)')
axes[0].set_ylabel('Predicted CLV (₹)')
axes[0].grid(True, alpha=0.3)

# Actual vs Predicted (Test)
axes[1].scatter(y_test, y_pred_test, alpha=0.6, color='green')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1].set_title(f'Test Set (R² = {test_r2:.3f})')
axes[1].set_xlabel('Actual CLV (₹)')
axes[1].set_ylabel('Predicted CLV (₹)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Business Insights and Recommendations

In [ ]:
# Generate comprehensive business insights
print("💼 CUSTOMER LIFETIME VALUE ANALYSIS REPORT")
print("=" * 80)

# Overall CLV Statistics
print("\n📊 OVERALL CLV STATISTICS")
print("-" * 40)
print(f"Total Customers Analyzed: {len(df):,}")
print(f"Average CLV: ₹{df['simple_clv'].mean():,.2f}")
print(f"Median CLV: ₹{df['simple_clv'].median():,.2f}")
print(f"Total Portfolio CLV: ₹{df['simple_clv'].sum():,.2f}")
print(f"Average Monthly Profit per Customer: ₹{df['monthly_profit'].mean():,.2f}")
print(f"Standard Deviation of CLV: ₹{df['simple_clv'].std():,.2f}")

# Top Value Customers
print("\n💎 TOP 10 VALUE CUSTOMERS")
print("-" * 40)
top_customers = df.nlargest(10, 'simple_clv')[['customer_id', 'simple_clv', 'monthly_profit', 'segment_name']]
top_customers.columns = ['Customer_ID', 'CLV', 'Monthly_Profit', 'Segment']
display(top_customers.round(2))

# Customer Segments Analysis
print("\n🎯 DETAILED SEGMENT ANALYSIS")
print("-" * 40)
segment_details = df.groupby('segment_name').agg({
    'customer_id': 'count',
    'simple_clv': ['mean', 'sum', 'std'],
    'monthly_profit': ['mean', 'sum'],
    'churn_probability': 'mean',
    'avg_monthly_spend': 'mean',
    'tenure_months': 'mean'
}).round(2)

# Flatten column names
segment_details.columns = ['Customer_Count', 'Avg_CLV', 'Total_CLV', 'CLV_StdDev', 
                          'Avg_Monthly_Profit', 'Total_Monthly_Profit', 'Avg_Churn_Risk',
                          'Avg_Monthly_Spend', 'Avg_Tenure_Months']

segment_details = segment_details.sort_values('Total_CLV', ascending=False)
display(segment_details)

In [ ]:
# Risk Analysis
print("⚠️ RISK ANALYSIS")
print("-" * 40)

# High risk customers (>50% churn probability)
high_risk_customers = df[df['churn_probability'] > 0.5]
print(f"High Risk Customers (>50% churn probability): {len(high_risk_customers):,}")
print(f"CLV at Risk: ₹{high_risk_customers['simple_clv'].sum():,.2f}")
print(f"Average CLV of High Risk Customers: ₹{high_risk_customers['simple_clv'].mean():,.2f}")
print(f"Percentage of Portfolio at Risk: {len(high_risk_customers)/len(df)*100:.1f}%")

# High value, high risk customers (priority for retention)
high_value_high_risk = df[(df['simple_clv'] > df['simple_clv'].quantile(0.7)) & 
                         (df['churn_probability'] > 0.3)]
print(f"\nHigh Value, High Risk Customers: {len(high_value_high_risk):,}")
print(f"CLV at High Risk: ₹{high_value_high_risk['simple_clv'].sum():,.2f}")

# Profitability Analysis
print("\n💰 PROFITABILITY ANALYSIS")
print("-" * 40)
profitable_customers = df[df['monthly_profit'] > 0]
unprofitable_customers = df[df['monthly_profit'] <= 0]

print(f"Profitable Customers: {len(profitable_customers):,} ({len(profitable_customers)/len(df)*100:.1f}%)")
print(f"Unprofitable Customers: {len(unprofitable_customers):,} ({len(unprofitable_customers)/len(df)*100:.1f}%)")
print(f"Average Profit of Profitable Customers: ₹{profitable_customers['monthly_profit'].mean():,.2f}")
if len(unprofitable_customers) > 0:
    print(f"Average Loss of Unprofitable Customers: ₹{unprofitable_customers['monthly_profit'].mean():,.2f}")

# Revenue breakdown
print("\n💰 REVENUE STREAM ANALYSIS")
print("-" * 40)
total_interchange = df['interchange_revenue'].sum()
total_interest = df['interest_revenue'].sum()
total_fees = df['annual_fee'].sum() / 12  # Monthly equivalent
total_revenue = total_interchange + total_interest + total_fees

print(f"Monthly interchange revenue: ₹{total_interchange:,.2f} ({total_interchange/total_revenue*100:.1f}%)")
print(f"Monthly interest revenue: ₹{total_interest:,.2f} ({total_interest/total_revenue*100:.1f}%)")
print(f"Monthly fee revenue: ₹{total_fees:,.2f} ({total_fees/total_revenue*100:.1f}%)")
print(f"Total monthly revenue: ₹{total_revenue:,.2f}")

In [ ]:
# Management Recommendations
print("🎯 STRATEGIC RECOMMENDATIONS FOR BANK MANAGEMENT")
print("=" * 80)

print("\n1. 🎯 CUSTOMER RETENTION STRATEGIES")
print("-" * 50)
print("• Focus retention efforts on high-value, high-risk customers")
print(f"• Priority customers for retention: {len(high_value_high_risk):,} customers worth ₹{high_value_high_risk['simple_clv'].sum():,.0f}")
print("• Implement targeted offers for customers with churn probability > 30%")
print("• Develop loyalty programs for Champions and High Value segments")
print("• Create personalized communication strategies for each segment")

print("\n2. 📈 CUSTOMER ACQUISITION STRATEGIES")
print("-" * 50)
champions = df[df['segment_name'] == 'Champions'] if 'Champions' in df['segment_name'].values else pd.DataFrame()
if not champions.empty:
    print(f"• Target customer profiles similar to Champions (avg CLV: ₹{champions['simple_clv'].mean():,.0f})")
    print(f"• Champions profile: Age {champions['age'].mean():.0f}, Income ₹{champions['annual_income'].mean():,.0f}")
print("• Optimize acquisition costs based on predicted CLV")
print("• Focus on customers with high monthly profit potential")
print("• Use predictive model to score prospects before acquisition")

print("\n3. 💳 PRODUCT AND PRICING STRATEGIES")
print("-" * 50)
print("• Cross-sell additional products to high CLV customers")
print("• Review pricing for unprofitable customer segments")
print("• Develop premium products for high-value segments")
print("• Implement dynamic pricing based on customer value")
print("• Consider segment-specific product features and benefits")

print("\n4. ⚠️ RISK MANAGEMENT")
print("-" * 50)
print("• Monitor customers with declining payment ratios")
print("• Implement early warning systems for churn risk")
print("• Consider credit limit adjustments for high-risk customers")
print(f"• Focus on {len(high_risk_customers):,} high-risk customers representing ₹{high_risk_customers['simple_clv'].sum():,.0f} in CLV")
print("• Develop risk-adjusted pricing models")

print("\n5. 📊 OPERATIONAL IMPROVEMENTS")
print("-" * 50)
print("• Implement real-time CLV scoring in CRM systems")
print("• Create automated alerts for customer value changes")
print("• Develop segment-specific service levels")
print("• Train customer service teams on segment characteristics")
print("• Establish CLV-based performance metrics for teams")

In [ ]:
# ROI Analysis for potential campaigns
print("\n📈 POTENTIAL ROI SCENARIOS")
print("-" * 40)

if len(high_risk_customers) > 0:
    retention_cost = 500  # Assumed cost per customer for retention campaign
    retention_rate_improvement = 0.2  # 20% improvement in retention
    
    potential_clv_saved = high_risk_customers['simple_clv'].sum() * retention_rate_improvement
    campaign_cost = len(high_risk_customers) * retention_cost
    roi = (potential_clv_saved - campaign_cost) / campaign_cost * 100
    
    print(f"Retention Campaign ROI Analysis:")
    print(f"  • Target Customers: {len(high_risk_customers):,}")
    print(f"  • Campaign Cost: ₹{campaign_cost:,.2f}")
    print(f"  • Potential CLV Saved: ₹{potential_clv_saved:,.2f}")
    print(f"  • Estimated ROI: {roi:.1f}%")
    print(f"  • Break-even retention rate: {campaign_cost/high_risk_customers['simple_clv'].sum()*100:.1f}%")

# Cross-selling opportunity
high_value_low_spend = df[(df['simple_clv'] > df['simple_clv'].median()) & 
                         (df['avg_monthly_spend'] < df['avg_monthly_spend'].median())]

if len(high_value_low_spend) > 0:
    print(f"\nCross-selling Campaign ROI Analysis:")
    cross_sell_cost = 200  # Cost per customer for cross-selling campaign
    avg_revenue_increase = 300  # Average monthly revenue increase from cross-selling
    success_rate = 0.15  # 15% success rate
    
    campaign_cost_cross = len(high_value_low_spend) * cross_sell_cost
    annual_revenue_increase = len(high_value_low_spend) * success_rate * avg_revenue_increase * 12
    roi_cross = (annual_revenue_increase - campaign_cost_cross) / campaign_cost_cross * 100
    
    print(f"  • Target Customers: {len(high_value_low_spend):,}")
    print(f"  • Campaign Cost: ₹{campaign_cost_cross:,.2f}")
    print(f"  • Estimated Annual Revenue Increase: ₹{annual_revenue_increase:,.2f}")
    print(f"  • Estimated ROI: {roi_cross:.1f}%")

print("\n" + "=" * 80)

## 9. Save Results and Export

In [ ]:
# Save comprehensive results
print("💾 SAVING ANALYSIS RESULTS")
print("=" * 50)

# Create comprehensive results dataframe
results_df = df[['customer_id', 'age', 'annual_income', 'card_type', 'tenure_months',
                'monthly_profit', 'monthly_revenue', 'monthly_cost', 'avg_monthly_spend',
                'churn_probability', 'historical_clv', 'simple_clv', 'npv_clv', 'predictive_clv',
                'segment', 'segment_name', 'is_active']].copy()

# Save main results
results_df.to_csv('clv_analysis_results.csv', index=False)
print(f"✅ Main results saved: clv_analysis_results.csv ({len(results_df):,} records)")

# Save segment summary
segment_details.to_csv('segment_analysis.csv')
print(f"✅ Segment analysis saved: segment_analysis.csv")

# Save high-risk customers for targeted campaigns
high_risk_export = high_risk_customers[['customer_id', 'simple_clv', 'monthly_profit', 
                                       'churn_probability', 'segment_name']].copy()
high_risk_export.to_csv('high_risk_customers.csv', index=False)
print(f"✅ High-risk customers saved: high_risk_customers.csv ({len(high_risk_export):,} records)")

# Save top customers for VIP treatment
top_customers_export = df.nlargest(100, 'simple_clv')[['customer_id', 'simple_clv', 'monthly_profit',
                                                      'segment_name', 'churn_probability']].copy()
top_customers_export.to_csv('top_customers.csv', index=False)
print(f"✅ Top customers saved: top_customers.csv (100 records)")

# Save model features importance
feature_importance.to_csv('feature_importance.csv', index=False)
print(f"✅ Feature importance saved: feature_importance.csv")

print("\n📁 Generated Files:")
files = ['clv_analysis_results.csv', 'segment_analysis.csv', 'high_risk_customers.csv', 
         'top_customers.csv', 'feature_importance.csv']

for file in files:
    if os.path.exists(file):
        size = os.path.getsize(file) / 1024
        print(f"  • {file}: {size:.1f} KB")

## 10. Interactive Analysis (Experiment Zone)

In [ ]:
# Interactive analysis section - modify these parameters to explore different scenarios
print("🧪 INTERACTIVE EXPERIMENT ZONE")
print("=" * 50)
print("Modify the parameters below to explore different scenarios:")

# Experiment parameters
EXPERIMENT_CONFIG = {
    'clv_threshold': 50000,  # Define high-value customers
    'churn_threshold': 0.3,  # Define high-risk customers
    'profit_threshold': 1000,  # Define profitable customers
    'discount_rate_sensitivity': [0.05, 0.10, 0.15, 0.20]  # Test different discount rates
}

print(f"\n🎯 Current experiment configuration:")
for key, value in EXPERIMENT_CONFIG.items():
    print(f"  {key}: {value}")

# Analyze impact of different CLV thresholds
high_value_customers = df[df['simple_clv'] > EXPERIMENT_CONFIG['clv_threshold']]
high_risk_customers_exp = df[df['churn_probability'] > EXPERIMENT_CONFIG['churn_threshold']]
profitable_customers_exp = df[df['monthly_profit'] > EXPERIMENT_CONFIG['profit_threshold']]

print(f"\n📊 Experiment Results:")
print(f"High-value customers (CLV > ₹{EXPERIMENT_CONFIG['clv_threshold']:,}): {len(high_value_customers):,} ({len(high_value_customers)/len(df)*100:.1f}%)")
print(f"High-risk customers (churn > {EXPERIMENT_CONFIG['churn_threshold']}): {len(high_risk_customers_exp):,} ({len(high_risk_customers_exp)/len(df)*100:.1f}%)")
print(f"Highly profitable customers (profit > ₹{EXPERIMENT_CONFIG['profit_threshold']}): {len(profitable_customers_exp):,} ({len(profitable_customers_exp)/len(df)*100:.1f}%)")

# Discount rate sensitivity analysis
print(f"\n💰 Discount Rate Sensitivity Analysis:")
print("-" * 40)
for rate in EXPERIMENT_CONFIG['discount_rate_sensitivity']:
    npv_test = calculate_npv_clv(df, rate, CLV_CONFIG['prediction_months'])
    print(f"Discount rate {rate:.1%}: Average NPV CLV = ₹{npv_test.mean():,.2f}")

print("\n💡 Try modifying the EXPERIMENT_CONFIG dictionary above and re-running this cell!")

## 11. Summary and Next Steps

In [ ]:
print("🎯 ANALYSIS SUMMARY")
print("=" * 50)
print("✅ Customer Lifetime Value analysis completed successfully!")
print(f"✅ Analyzed {len(df):,} customers across {len(df.columns)} features")
print(f"✅ Built predictive model with R² = {test_r2:.3f}")
print(f"✅ Segmented customers into {df['segment'].nunique()} meaningful groups")
print(f"✅ Identified {len(high_risk_customers):,} high-risk customers")
print(f"✅ Generated comprehensive business recommendations")

print("\n🎓 WHAT YOU LEARNED")
print("-" * 30)
print("• Multiple CLV calculation methodologies")
print("• Machine learning for customer value prediction")
print("• Customer segmentation techniques")
print("• Churn probability estimation")
print("• Business insight generation")
print("• ROI analysis for marketing campaigns")

print("\n🏦 BANKING APPLICATIONS")
print("-" * 30)
print("• Customer acquisition optimization")
print("• Retention campaign targeting")
print("• Product development strategies")
print("• Risk-based pricing")
print("• Portfolio management")
print("• Resource allocation")

print("\n🔗 USEFUL LINKS")
print("-" * 30)
print("• GitHub Repository: https://github.com/VinayaSharada/FinancialAnalyticsCourse")
print("• Data Generation Notebook: clv_data_generation.ipynb")
print("• Course Materials: Financial Analytics Course")

print("\n🚀 NEXT STEPS")
print("-" * 30)
print("1. Implement findings in production systems")
print("2. Set up automated model retraining")
print("3. Create management dashboards")
print("4. Design targeted marketing campaigns")
print("5. Monitor model performance over time")
print("6. Expand analysis to other product lines")

print("\n" + "=" * 50)
print("📊 Customer Lifetime Value Analysis Complete! 📊")
print("=" * 50)